# IMDB Sentiment Classification using MLP
Student: Oumayma Chaouch  
Course: CSC3348 01
Dataset: IMDB 50K Reviews  
Feature Engineering: VADER + TextBlob  
Model: Multi-Layer Perceptron



In [ ]:
!pip install vaderSentiment textblob


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 2.7 MB/s eta 0:00:00


In [1]:
import pandas as pd
import numpy as np

# Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Deep Learning
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.layers import Embedding, LSTM, GRU, Conv1D, GlobalMaxPooling1D
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Metrics
from sklearn.metrics import accuracy_score, classification_report

In [2]:
from google.colab import files
uploaded = files.upload()


Saving IMDB Dataset.csv to IMDB Dataset.csv


In [7]:



df = pd.read_csv("IMDB Dataset.csv")
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})

In [10]:
import nltk
nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


True

In [12]:
import nltk
nltk.download('vader_lexicon')

from nltk.sentiment import SentimentIntensityAnalyzer
from textblob import TextBlob

sia = SentimentIntensityAnalyzer()

# VADER
df['v_scores'] = df['review'].apply(lambda x: sia.polarity_scores(x))
df['v_neg'] = df['v_scores'].apply(lambda x: x['neg'])
df['v_neu'] = df['v_scores'].apply(lambda x: x['neu'])
df['v_pos'] = df['v_scores'].apply(lambda x: x['pos'])
df['v_comp'] = df['v_scores'].apply(lambda x: x['compound'])

# TextBlob
df['tb_polarity'] = df['review'].apply(lambda x: TextBlob(x).sentiment.polarity)
df['tb_subjectivity'] = df['review'].apply(lambda x: TextBlob(x).sentiment.subjectivity)

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [13]:
X_features = df[['v_neg','v_neu','v_pos','v_comp',
                 'tb_polarity','tb_subjectivity']]

y = df['sentiment']

In [14]:
X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_features, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_f = scaler.fit_transform(X_train_f)
X_test_f = scaler.transform(X_test_f)

MODEL 1: IMPROVED MLP


In [15]:
model_mlp = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_f.shape[1],)),
    BatchNormalization(),
    Dropout(0.5),

    Dense(32, activation='relu'),
    Dropout(0.3),

    Dense(1, activation='sigmoid')
])

model_mlp.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [16]:
history_mlp = model_mlp.fit(
    X_train_f, y_train_f,
    epochs=10,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/10
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - accuracy: 0.7354 - loss: 0.5511 - val_accuracy: 0.7760 - val_loss: 0.4790
Epoch 2/10
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.7613 - loss: 0.5067 - val_accuracy: 0.7770 - val_loss: 0.4762
Epoch 3/10
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.7641 - loss: 0.4979 - val_accuracy: 0.7774 - val_loss: 0.4755
Epoch 4/10
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.7648 - loss: 0.4955 - val_accuracy: 0.7799 - val_loss: 0.4762
Epoch 5/10
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.7652 - loss: 0.4967 - val_accuracy: 0.7788 - val_loss: 0.4756
Epoch 6/10
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.7658 - loss: 0.4956 - val_accuracy: 0.7782 - val_loss: 0.4780
Epoch 7/10
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.7667 - loss: 0.4943 - val_accuracy: 0.7782 - val_loss: 0.4746
Epoch 8/10
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.7658 - loss: 0.4929 - 

In [17]:
y_pred_mlp = (model_mlp.predict(X_test_f) > 0.5).astype("int32")

print("MLP Accuracy:", accuracy_score(y_test_f, y_pred_mlp))
print(classification_report(y_test_f, y_pred_mlp))

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
MLP Accuracy: 0.7776
              precision    recall  f1-score   support

           0       0.77      0.79      0.78      4961
           1       0.79      0.77      0.78      5039

    accuracy                           0.78     10000
   macro avg       0.78      0.78      0.78     10000
weighted avg       0.78      0.78      0.78     10000



In [18]:
max_words = 10000
max_len = 200

tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(df['review'])

X_seq = tokenizer.texts_to_sequences(df['review'])
X_pad = pad_sequences(X_seq, maxlen=max_len)

y = df['sentiment']

In [19]:
X_train, X_test, y_train, y_test = train_test_split(
    X_pad, y, test_size=0.2, random_state=42
)

MODEL 2: LSTM

In [20]:
model_lstm = Sequential([
    Embedding(input_dim=max_words, output_dim=128, input_length=max_len),
    LSTM(128, dropout=0.2, recurrent_dropout=0.2),
    Dense(1, activation='sigmoid')
])

model_lstm.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [21]:
history_lstm = model_lstm.fit(
    X_train, y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 347s 677ms/step - accuracy: 0.7894 - loss: 0.4586 - val_accuracy: 0.8533 - val_loss: 0.3423
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 350s 615ms/step - accuracy: 0.8716 - loss: 0.3137 - val_accuracy: 0.8726 - val_loss: 0.3346
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 330s 631ms/step - accuracy: 0.8971 - loss: 0.2612 - val_accuracy: 0.8620 - val_loss: 0.3662
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 326s 639ms/step - accuracy: 0.9179 - loss: 0.2160 - val_accuracy: 0.8506 - val_loss: 0.3530
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 312s 624ms/step - accuracy: 0.9165 - loss: 0.2139 - val_accuracy: 0.8649 - val_loss: 0.3520


In [22]:
y_pred_lstm = (model_lstm.predict(X_test) > 0.5).astype("int32")

print("LSTM Accuracy:", accuracy_score(y_test, y_pred_lstm))
print(classification_report(y_test, y_pred_lstm))

313/313 ━━━━━━━━━━━━━━━━━━━━ 31s 96ms/step
LSTM Accuracy: 0.8708
              precision    recall  f1-score   support

           0       0.84      0.92      0.88      4961
           1       0.91      0.83      0.87      5039

    accuracy                           0.87     10000
   macro avg       0.87      0.87      0.87     10000
weighted avg       0.87      0.87      0.87     10000



MODEL 3: GRU

In [23]:
model_gru = Sequential([
    Embedding(input_dim=max_words, output_dim=128, input_length=max_len),
    GRU(128),
    Dense(1, activation='sigmoid')
])

model_gru.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [24]:
history_gru = model_gru.fit(
    X_train, y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 322s 636ms/step - accuracy: 0.8045 - loss: 0.4203 - val_accuracy: 0.8514 - val_loss: 0.3333
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 232s 464ms/step - accuracy: 0.9059 - loss: 0.2428 - val_accuracy: 0.8938 - val_loss: 0.2683
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 267s 474ms/step - accuracy: 0.9377 - loss: 0.1687 - val_accuracy: 0.8864 - val_loss: 0.2836
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 235s 470ms/step - accuracy: 0.9638 - loss: 0.1052 - val_accuracy: 0.8701 - val_loss: 0.3847
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 239s 478ms/step - accuracy: 0.9762 - loss: 0.0717 - val_accuracy: 0.8838 - val_loss: 0.3987


In [26]:
y_pred_gru = (model_gru.predict(X_test) > 0.5).astype("int32")

print("GRU Accuracy:", accuracy_score(y_test, y_pred_gru))
print(classification_report(y_test, y_pred_gru))

313/313 ━━━━━━━━━━━━━━━━━━━━ 18s 55ms/step
GRU Accuracy: 0.8826
              precision    recall  f1-score   support

           0       0.90      0.86      0.88      4961
           1       0.87      0.91      0.89      5039

    accuracy                           0.88     10000
   macro avg       0.88      0.88      0.88     10000
weighted avg       0.88      0.88      0.88     10000



In [ ]:
gru_folder = os.path.join(base_folder, "GRU")
os.makedirs(gru_folder, exist_ok=True)

gru_model.save(os.path.join(gru_folder, "gru_model.h5"))
print("GRU model saved successfully at:", gru_folder)

MODEL4: CNN

In [25]:
model_cnn = Sequential([
    Embedding(input_dim=max_words, output_dim=128, input_length=max_len),
    Conv1D(128, 5, activation='relu'),
    GlobalMaxPooling1D(),
    Dense(1, activation='sigmoid')
])

model_cnn.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [27]:
history_cnn = model_cnn.fit(
    X_train, y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 112s 220ms/step - accuracy: 0.8141 - loss: 0.4016 - val_accuracy: 0.8832 - val_loss: 0.2733
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 90s 181ms/step - accuracy: 0.9198 - loss: 0.2059 - val_accuracy: 0.8944 - val_loss: 0.2533
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 143s 183ms/step - accuracy: 0.9665 - loss: 0.1075 - val_accuracy: 0.8956 - val_loss: 0.2729
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 146s 190ms/step - accuracy: 0.9897 - loss: 0.0455 - val_accuracy: 0.8920 - val_loss: 0.3088
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 140s 187ms/step - accuracy: 0.9985 - loss: 0.0154 - val_accuracy: 0.8928 - val_loss: 0.3533


In [31]:
y_pred_cnn = (model_cnn.predict(X_test) > 0.5).astype("int32")

print("CNN Accuracy:", accuracy_score(y_test, y_pred_cnn))
print(classification_report(y_test, y_pred_cnn))

313/313 ━━━━━━━━━━━━━━━━━━━━ 16s 51ms/step
CNN Accuracy: 0.8902
              precision    recall  f1-score   support

           0       0.91      0.87      0.89      4961
           1       0.87      0.91      0.89      5039

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000



Word2Vec


In [30]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 31.1 MB/s eta 0:00:00


In [32]:
from gensim.models import Word2Vec

sentences = [review.split() for review in df['review']]
w2v_model = Word2Vec(sentences, vector_size=100, window=5, min_count=2)

In [36]:

import pandas as pd
import numpy as np
from gensim.models import Word2Vec
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

In [37]:
df = pd.read_csv("IMDB Dataset.csv")
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})

In [38]:
max_words = 10000
max_len = 200

tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(df['review'])

X_seq = tokenizer.texts_to_sequences(df['review'])
X_pad = pad_sequences(X_seq, maxlen=max_len)

y = df['sentiment']

In [39]:
X_train, X_test, y_train, y_test = train_test_split(
    X_pad, y, test_size=0.2, random_state=42
)

In [40]:
sentences = [review.split() for review in df['review']]
w2v_model = Word2Vec(sentences, vector_size=100, window=5, min_count=2)

In [41]:
embedding_dim = 100
vocab_size = min(max_words, len(tokenizer.word_index) + 1)

embedding_matrix = np.zeros((vocab_size, embedding_dim))
for word, i in tokenizer.word_index.items():
    if i >= vocab_size:
        continue
    if word in w2v_model.wv:
        embedding_matrix[i] = w2v_model.wv[word]

In [42]:
model_lstm_w2v = Sequential([
    Embedding(input_dim=vocab_size,
              output_dim=embedding_dim,
              weights=[embedding_matrix],
              input_length=max_len,
              trainable=False),  # Change to True to fine-tune
    LSTM(128, dropout=0.2, recurrent_dropout=0.2),
    Dense(1, activation='sigmoid')
])

model_lstm_w2v.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [43]:
history_lstm_w2v = model_lstm_w2v.fit(
    X_train, y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 332s 656ms/step - accuracy: 0.7670 - loss: 0.4783 - val_accuracy: 0.8571 - val_loss: 0.3268
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 266s 532ms/step - accuracy: 0.8564 - loss: 0.3339 - val_accuracy: 0.8789 - val_loss: 0.2809
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 262s 524ms/step - accuracy: 0.8751 - loss: 0.2942 - val_accuracy: 0.8923 - val_loss: 0.2639
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 262s 524ms/step - accuracy: 0.8878 - loss: 0.2673 - val_accuracy: 0.8888 - val_loss: 0.2616
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 260s 521ms/step - accuracy: 0.8952 - loss: 0.2529 - val_accuracy: 0.9003 - val_loss: 0.2427


In [45]:
y_pred = (model_lstm_w2v.predict(X_test) > 0.5).astype("int32")

print("LSTM + Word2Vec Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

313/313 ━━━━━━━━━━━━━━━━━━━━ 28s 90ms/step
LSTM + Word2Vec Accuracy: 0.9032
              precision    recall  f1-score   support

           0       0.91      0.90      0.90      4961
           1       0.90      0.91      0.90      5039

    accuracy                           0.90     10000
   macro avg       0.90      0.90      0.90     10000
weighted avg       0.90      0.90      0.90     10000



SAVE MODELS


In [51]:
import os
from google.colab import files

# Create a base folder for all models
base_folder = "/content/assignment2_models"
os.makedirs(base_folder, exist_ok=True)

# Dictionary of your models
models = {
    "CNN": model_cnn,
    "LSTM": model_lstm,
    "GRU": model_gru,
    "MLP": model_mlp,
    "LSTM_Word2Vec": model_lstm_w2v
}

# Save each model and trigger download
for name, model in models.items():
    folder = os.path.join(base_folder, name)
    os.makedirs(folder, exist_ok=True)

    # Save in native Keras format
    file_path = os.path.join(folder, f"{name.lower()}_model.keras")
    model.save(file_path)

    # Trigger download
    print(f"Preparing download for {name} model...")
    files.download(file_path)

Preparing download for CNN model...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Preparing download for LSTM model...


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Preparing download for LSTM_Word2Vec model...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Baseline Model: Logistic Regression

To evaluate whether the nonlinear MLP architecture provides additional predictive power, we compare it with a simpler linear classifier (Logistic Regression) using the same lexicon-based features.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

print("Logistic Regression Results:")
print(classification_report(y_test, y_pred_lr))


Logistic Regression Results:
              precision    recall  f1-score   support

           0       0.77      0.78      0.77      4961
           1       0.78      0.77      0.78      5039

    accuracy                           0.78     10000
   macro avg       0.78      0.78      0.78     10000
weighted avg       0.78      0.78      0.78     10000



## Testing on Custom Reviews


In [ ]:
loss, accuracy = model.evaluate(X_test, y_test)
print("Test Accuracy:", accuracy)


313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.7788 - loss: 0.4694
Test Accuracy: 0.7775999903678894


In [ ]:
new_review = "This movie was absolutely fantastic, I loved every minute of it."


In [ ]:
# VADER
v_scores = analyzer.polarity_scores(new_review)

# TextBlob
tb = TextBlob(new_review)

feature_cols = ['v_neg','v_neu','v_pos','v_comp',
                'tb_polarity','tb_subjectivity']

features_df = pd.DataFrame([[
    v_scores['neg'],
    v_scores['neu'],
    v_scores['pos'],
    v_scores['compound'],
    tb.sentiment.polarity,
    tb.sentiment.subjectivity
]], columns=feature_cols)


In [ ]:
features_scaled = scaler.transform(features_df)


In [ ]:
prediction = model.predict(features_scaled)

print("Probability positive:", prediction[0][0])

if prediction[0][0] > 0.5:
    print("Predicted sentiment: Positive")
else:
    print("Predicted sentiment: Negative")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
Probability positive: 0.9976298
Predicted sentiment: Positive


In [ ]:
from tensorflow.keras.models import load_model

loaded_model = load_model("mlp_model.keras")

loss, accuracy = loaded_model.evaluate(X_test, y_test)
print("Loaded model accuracy:", accuracy)


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 8 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.7788 - loss: 0.4694
Loaded model accuracy: 0.7775999903678894
